## Backstage

[Backstage.io](https://backstage.io) ist eine Open-Source-Developer-Plattform, die von Spotify entwickelt wurde, um die Verwaltung von Softwareprojekten zu vereinfachen. 

Mit Backstage können Unternehmen eine zentrale Anlaufstelle für all ihre internen Tools, Services, Dokumentationen und Software-Komponenten schaffen. 

Im Mittelpunkt steht das Konzept des "Service Catalogs", der alle Applikationen und Services übersichtlich darstellt. 

Durch Plugins lässt sich Backstage flexibel erweitern und an individuelle Bedürfnisse anpassen. Ziel ist es, Entwickler:innen die tägliche Arbeit zu erleichtern und eine einheitliche Nutzererfahrung über verschiedene Tools hinweg zu bieten.


### Installation

Dazu greifen wir auf Scripts aus dem [Lern Cloud Projekt](https://github.com/mc-b/lerncloud/tree/main/services) zurück, dass
* Installiert nvm und Node.js
* Entpackt eine vorinstallierte Backstage Umgebung nach `~/backstage`


In [ ]:
%%bash
curl -sfL https://raw.githubusercontent.com/mc-b/lerncloud/refs/heads/main/services/backstage.sh | bash -

### Backstage UI

Nach dem Ausführen der untenstehenden Schritte ist das Backstage UI unter folgendem URL erreichbar:

In [ ]:
%%bash
echo "http://$(cat ~/work/server-ip):3000"

### Konfiguration anpassen

Die Konfiguration kann unter folgenden URLs angepasst werden, z.B. um weitere Catalog-Entries hinzuzufügen.

* [backstage/app-config.yaml](../../../edit/backstage/app-config.yaml)
* [backstage/examples/org.yaml](../../../edit/backstage/examples/org.yaml)


### Einrichten der Authentifizierung

Für Backstage stehen Ihnen verschiedene Authentifizierungsanbieter zur Verfügung. Hier verwenden wir GitHub.

**Fügt in GitHub eine neue App hinzu**

Geht zu [https://github.com/settings/applications/new](https://github.com/settings/applications/new), um Eure OAuth-App zu erstellen.


In [ ]:
%%bash
echo "Backstage-Frontend        : http://$(cat ~/work/server-ip):3000"
echo "Authorization callback URL: http://$(cat ~/work/server-ip):7007/api/auth/github/handler/frame"

### Einrichten Personal access tokens (classic)

Um neue Repositories erstellen zu können, brauchen wir einen Personal access tokens (classic).

Geht zu [https://github.com/settings/tokens](https://github.com/settings/tokens) und erstellt einen Classic Token.

Dieser braucht mindestens folgende Rechte:

* Reading software components:
    * repo
* Reading organization data:
    * read:org
    * read:user
    * user:email
* Publishing software templates:
    * repo
    * workflow (if templates include GitHub workflows)  


### Setzen der Umgebung

Setzt die erstellen Token etc. als Umgebungsvariablen

In [ ]:
import os
os.environ['BACKSTAGE_ORG']='Auto Shop Group'
os.environ['GITHUB_CLIENT_ID']=''
os.environ['GITHUB_SECRET']=''
os.environ['GITHUB_TOKEN']=''


### Startet Backstage

Die Umgebung kann mittels des **Stopp** Buttons wieder bendet werden.

Es wird eine In-Memory Datenbank verwendet.

In [ ]:
%%bash
cd ~/backstage
source ~/.nvm/nvm.sh 
export NODE_OPTIONS=--no-node-snapshot
export BACKSTAGE_HOST="$(cat ~/work/server-ip)"
yarn start


- - -
### Kubernetes Integration (optional)

Dazu Erstellen wir einen Service Account mit Cluster Role Binding `view`.

In [ ]:
%%bash
kubectl create serviceaccount backstage 
kubectl create clusterrolebinding backstage-view --clusterrole=view --serviceaccount=default:backstage

Zusätzlich brauchen wir den Metrics Service

In [ ]:
%%bash
kubectl apply -f https://github.com/kubernetes-sigs/metrics-server/releases/latest/download/components.yaml && \
kubectl patch deployment metrics-server -n kube-system \
  --type='json' \
  -p='[{"op":"add","path":"/spec/template/spec/containers/0/args/-","value":"--kubelet-insecure-tls"}]'


Erweitern die Kubernetes Ressourcen Deployments, ReplicaSet und Pod um den Label `backstage.io/kubernetes-id`:

    kind: Deployment
    apiVersion: apps/v1
    metadata:
      name: catalog
      labels:
        app: catalog
        backstage.io/kubernetes-id: catalog

Stellen mittels einer Annotation `backstage.io/kubernetes-id` in `catalog-info.yaml` den Bezug zur Kubernetes Ressource her:

    # catalog
    apiVersion: backstage.io/v1alpha1
    kind: Component
    metadata:
      name: shop-catalog
      description: Catalog service for managing product listings.
      annotations:
        backstage.io/kubernetes-id: catalog   


Und Erweitern [app-config.yaml](../../../edit/backstage/app-config.yaml) wie folgt:

    kubernetes:
      serviceLocatorMethod:
        type: multiTenant
      clusterLocatorMethods:
        - type: config
          clusters:
            - name: kind-kind
              url: https://127.0.0.1:38435
              authProvider: serviceAccount
              serviceAccountToken: ${K8S_SERVICE_ACCOUNT_TOKEN}
              caData: ${K8S_CA_DATA}
              skipTLSVerify: false  
              
Die zwei Umgebungsvariablen setzen wir vor dem Start von Backstage          

In [ ]:
%%bash
export K8S_SERVICE_ACCOUNT_TOKEN=$(kubectl create token backstage --duration=24h)
export K8S_CA_DATA=$(kubectl config view --raw -o jsonpath="{.clusters[0].cluster.certificate-authority-data}")

cd ~/backstage
source ~/.nvm/nvm.sh 
export NODE_OPTIONS=--no-node-snapshot
export BACKSTAGE_HOST="$(cat ~/work/server-ip)"
yarn start

---
### Backstage als Service einrichten (optinal)



In [ ]:
%%bash
source ~/.nvm/nvm.sh 
mkdir -p ~/.config/systemd/user
cat <<EOF > ~/.config/systemd/user/backstage.service
[Unit]
Description=Backstage.io
After=network.target

[Service]
Environment="NODE_OPTIONS=--no-node-snapshot"
Environment="BACKSTAGE_ORG=${BACKSTAGE_ORG}"
Environment="BACKSTAGE_HOST=$(cat ~/work/server-ip)"
Environment="GITHUB_CLIENT_ID=${GITHUB_CLIENT_ID}"
Environment="GITHUB_SECRET=${GITHUB_SECRET}"
Environment="GITHUB_TOKEN=${GITHUB_TOKEN}"
Type=simple
WorkingDirectory=/home/ubuntu/backstage
ExecStartPre=/bin/bash -c 'source /home/ubuntu/.nvm/nvm.sh'
ExecStart=$(which node) $(which yarn) start
Restart=on-failure

[Install]
WantedBy=default.target
EOF
cat ~/.config/systemd/user/backstage.service

**Backstage Service starten**

    systemctl --user daemon-reload
    systemctl --user  start backstage


**Status überprüfen**

    systemctl --user status backstage.service

- - -

## Aufträge

* Erweitert Eure Git Repositories um `catalog-info.yaml` Dateien und integriert diese in Backstage
* Versucht ein Projekte oder Eure Firma in Backstage zu modelieren
* Erstellt ein Template welche z.B. Dynamisch eine `cloud-init.yaml` Datei zusammenstellt
* Installiert ein Plug-In von [https://backstage.io/plugins/](https://backstage.io/plugins/).




**Links**

* [Backstage.io](https://backstage.io)
* [Roadie Backstage Plug-Ins](https://roadie.io/backstage/plugins/)
* [Backstage Plug-Ins](https://backstage.io/plugins/)
* [Auto Shop Backstage](https://gitlab.com/ch-mc-b/autoshop-ms/infra/backstage)
* [lernmaas catalog-info.yaml](https://github.com/mc-b/lernmaas/blob/master/catalog-info.yaml)
* [lerncloud catalog-info.yaml](https://github.com/mc-b/lerncloud/blob/main/catalog-info.yaml)